# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [ ]:
"""Unit of analysis: One row represents the performance of one content item for one client on one report date.

Time window: I will use March 2026 (month = 2026-03) as my development window. I will not use the final month (June 2026) for developing or testing label logic because it should remain a sealed test month.
"""

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
"""
Table: I will use the daily content performance table, which contains daily performance data at the report_date × client × content grain.

Prediction / ranking target: I want to rank content items based on their future performance, using a future performance outcome as the label or proxy.

Features: I will use only information that is available at the decision moment and does not depend on the future outcome.

Context: Client, content, and report date identifiers will be kept as context to identify each observation, but they are not treated as predictive features.

Deliberately excluded: I will exclude label-derived or future-derived fields such as trend_pct and trend_direction from the honest feature set because they can leak information about the outcome into the model.

"""

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [6]:
!pip -q install duckdb huggingface_hub

import os
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN

con = duckdb.connect()

print("Setup complete")

Setup complete


In [7]:
from huggingface_hub import list_repo_files

files = list_repo_files(
    "FlyRank/internship-warehouse",
    repo_type="dataset",
    token=HF_TOKEN
)

for f in files:
    if "fact_content_daily_performance" in f and "2026-03" in f:
        print(f)

fact_content_daily_performance/month=2026-03/data_0.parquet


In [8]:
from huggingface_hub import hf_hub_download

march_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

print(march_file)

fact_content_daily_performance/month=202(…):   0%|          | 0.00/124M [00:00<?, ?B/s]

/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet


In [10]:
query1 = """
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS row_count
FROM read_parquet(?)
GROUP BY report_date, client_hash_id, content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
"""

result1 = con.execute(query1, [march_file]).df()
result1

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,row_count


In [11]:
query2 = """
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM read_parquet(?)
"""

result2 = con.execute(query2, [march_file]).df()
result2

,row_count,min_date,max_date
0,9841378,2026-03-01,2026-03-31


In [12]:
query3 = """
SELECT
    COUNT(*) AS available_rows
FROM read_parquet(?)
WHERE ga4_data_available IS TRUE
"""

result3 = con.execute(query3, [march_file]).df()
result3

,available_rows
0,413966


In [14]:
feature_frame = con.execute("""
SELECT
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_sessions,
    ga4_engaged_sessions
FROM read_parquet(?)
WHERE ga4_data_available IS TRUE
LIMIT 10000
""", [march_file]).df()

feature_frame.head()


,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions
0,0,0,NaN,1,0
1,0,0,NaN,1,0
2,0,0,NaN,1,0
3,0,0,NaN,1,0
4,0,0,NaN,1,0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
"""
1. gsc_impressions — Knowable at the decision moment because it is observed Search Console performance up to the current report date.
2. gsc_clicks — Knowable at the decision moment because it is observed Search Console click activity up to the current report date.
3. gsc_avg_position — Knowable at the decision moment because the content's search position has already been observed by the current report date.
4. ga4_sessions — Knowable at the decision moment because GA4 session data is already observed for rows where `ga4_data_available IS TRUE`.
5. ga4_engaged_sessions — Knowable at the decision moment because engaged-session activity has already been observed in GA4 by the current report date.

"""

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.